# Memory Optimization & Downcasting

## Definition
Large DataFrames can consume gigabytes of RAM. **Downcasting** reduces memory by using smaller numeric types
without losing significant precision.

### Memory Reduction Strategies
| Strategy | Potential Savings |
|----------|------------------|
| int64 → int8/int16 | 75–87% |
| float64 → float32 | 50% |
| object → category | Up to 95% (for repeated strings) |
| object → bool | 87% |
| Remove unused columns | 100% for that column |

### pandas dtype hierarchy (size)
```
int8 (1B) < int16 (2B) < int32 (4B) < int64 (8B)
float16 (2B) < float32 (4B) < float64 (8B)
bool (1B)
category ≈ int + lookup table (very small for repeated values)
```

In [ ]:
import pandas as pd
import numpy as np

# Create a large-ish DataFrame with default dtypes
rng = np.random.default_rng(42)
n = 500_000

df = pd.DataFrame({
    'user_id':    rng.integers(0, 100_000, n),         # int64
    'age':        rng.integers(18, 65, n),              # int64, max 65
    'score':      rng.uniform(0, 100, n),               # float64
    'gender':     rng.choice(['M', 'F'], n),            # object
    'dept':       rng.choice(['Eng', 'Mkt', 'HR', 'Finance', 'Legal'], n),  # object
    'is_active':  rng.choice([0, 1], n),               # int64
    'n_logins':   rng.integers(0, 1000, n),             # int64
    'revenue':    rng.normal(1000, 500, n)              # float64
})

def mem_usage(df):
    return df.memory_usage(deep=True).sum() / 1024**2

print(f'Initial memory: {mem_usage(df):.2f} MB')
print(df.dtypes)

In [ ]:
# Step 1: Downcast integers
def downcast_integers(df):
    int_cols = df.select_dtypes(include=['int64', 'int32']).columns
    df[int_cols] = df[int_cols].apply(pd.to_numeric, downcast='unsigned')
    return df

df_opt = df.copy()
df_opt = downcast_integers(df_opt)
print('After integer downcast:')
print(df_opt[['user_id', 'age', 'is_active', 'n_logins']].dtypes)
print(f'Memory: {mem_usage(df_opt):.2f} MB')

In [ ]:
# Step 2: Downcast floats
def downcast_floats(df):
    float_cols = df.select_dtypes(include=['float64']).columns
    df[float_cols] = df[float_cols].apply(pd.to_numeric, downcast='float')
    return df

df_opt = downcast_floats(df_opt)
print('After float downcast:')
print(df_opt[['score', 'revenue']].dtypes)
print(f'Memory: {mem_usage(df_opt):.2f} MB')

In [ ]:
# Step 3: Convert object to category
def convert_to_category(df, max_unique_ratio=0.5):
    obj_cols = df.select_dtypes(include=['object']).columns
    for col in obj_cols:
        n_unique = df[col].nunique()
        if n_unique / len(df) < max_unique_ratio:
            df[col] = df[col].astype('category')
    return df

df_opt = convert_to_category(df_opt)
print('After category conversion:')
print(df_opt[['gender', 'dept']].dtypes)
print(f'Memory: {mem_usage(df_opt):.2f} MB')

In [ ]:
# Step 4: Convert int to bool where appropriate
df_opt['is_active'] = df_opt['is_active'].astype(bool)
print(f'After bool conversion: {mem_usage(df_opt):.2f} MB')

print('\n=== SUMMARY ===')
print(f'Original: {mem_usage(df):.2f} MB')
print(f'Optimized: {mem_usage(df_opt):.2f} MB')
print(f'Savings: {(1 - mem_usage(df_opt)/mem_usage(df))*100:.1f}%')

In [ ]:
# All-in-one optimization function
def optimize_dtypes(df):
    df = df.copy()
    for col in df.columns:
        col_type = df[col].dtype
        if col_type == 'int64' or col_type == 'int32':
            df[col] = pd.to_numeric(df[col], downcast='unsigned')
        elif col_type == 'float64':
            df[col] = pd.to_numeric(df[col], downcast='float')
        elif col_type == 'object':
            n_unique_ratio = df[col].nunique() / len(df)
            if n_unique_ratio < 0.5:
                df[col] = df[col].astype('category')
    return df

df_final = optimize_dtypes(df)
print(f'One-shot optimization: {mem_usage(df):.1f} MB -> {mem_usage(df_final):.1f} MB '
      f'({(1-mem_usage(df_final)/mem_usage(df))*100:.0f}% reduction)')

In [ ]:
# Chunked reading for very large files
import io

# Simulate a large CSV file in memory
large_csv = 'a,b,c\n' + '\n'.join(f'{i},{i*2},{i*3}' for i in range(10000))

# Read in chunks — process without loading all into memory
total_sum = 0
for chunk in pd.read_csv(io.StringIO(large_csv), chunksize=1000):
    chunk['a'] = chunk['a'].astype(np.int16)   # optimize as we read
    total_sum += chunk['a'].sum()

print('Sum of column a (chunked):', total_sum)